# **Durability — RUL DATASET GENERATION**

## Prerequisite

Run [`01_generate_dataset.ipynb`](01_generate_dataset.ipynb) $\to$
[`02_train_pce.ipynb`](02_train_pce.ipynb) $\to$
[`03_generate_dataset_nn.ipynb`](03_generate_dataset_nn.ipynb) $\to$
[`04_train_nn.ipynb`](04_train_nn.ipynb) first — this notebook loads the global NN
(`lambda 1`/`lambda 2` vs. `(fck, rh, cov, t)`) that stage 4 writes.

## What this notebook does

Fixes a single design point `(fck, rh, cov)` and sweeps a list of time steps through the trained
NN, giving $\lambda_1(t)$ and $\lambda_2(t)$ at each one **directly** — no need to pick a
per-time-step PCE first. $\lambda_3$ and $\lambda_4$ barely move across the design space, which is
why they were never modelled by the NN — they're fixed here instead, to a value you set or, by
default, the mean over the NN's own training dataset.

At each time step, `generate_rul_dataset_durability` builds a `GlamFKML(lam1, lam2, lam3, lam4)` and
draws `n_glam_samples` Monte Carlo realisations of the state limit function $g$ = cover −
carbonation depth — the raw material for the spaghetti plot and the RUL analysis in
[`05_plot_rul_analysis.ipynb`](05_plot_rul_analysis.ipynb).

## 1. Libraries

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import dill
import numpy as np
import pandas as pd

from functions import *

/home/casa-wand/steam2tb/2024-1_victor_hugo_renata_maria/.venv/lib/python3.11/site-packages/UQpy/__init__.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


## 2. Config

`n_latent_samples`, `installation_year`, `co2_scenario`, `cement_type` and `exposure_conditions`
must match [`04_train_nn.ipynb`](04_train_nn.ipynb) — it names the NN model files being loaded.

`lambda3_fixed`/`lambda4_fixed` left as `None` fall back to the mean of `lambda 3`/`lambda 4` over
the NN training dataset; set either explicitly to override.

In [2]:
n_latent_samples    = 2500      # must match stage 4 — it names the NN model files being loaded
installation_year   = 1980
co2_scenario        = "SSP2-4.5"
cement_type         = 3
exposure_conditions = 2

fck_fixed = 25.0   # fixed compressive strength for the whole sweep (MPa)
rh_fixed  = 55.0   # fixed relative humidity for the whole sweep (%)
cov_fixed = 25.0   # fixed cover for the whole sweep (mm)
times     = np.linspace(0, 100, 20, endpoint=True)  # time steps to sweep through the NN — any grid, not just the training one

lambda3_fixed = 0.198312   # None = mean of lambda 3 over the NN training dataset
lambda4_fixed = 0.130039   # None = mean of lambda 4 over the NN training dataset

n_glam_samples = 50000   # Monte Carlo samples drawn from the GLD at each time step

print(f"Sweeping t = {times.min()}..{times.max()} years at fck={fck_fixed}, rh={rh_fixed}, cov={cov_fixed}")

Sweeping t = 0.0..100.0 years at fck=25.0, rh=55.0, cov=25.0


## 3. Predict lambda 1/2, fix lambda 3/4, and draw the GLD samples

In [3]:
print("="*60)
print("GENERATING THE RUL DATASET")
print("="*60)

result = generate_rul_dataset_durability(
                                            fck=fck_fixed,
                                            rh=rh_fixed,
                                            cov=cov_fixed,
                                            times=times,
                                            n_latent_samples=n_latent_samples,
                                            installation_year=installation_year,
                                            cement_type=cement_type,
                                            exposure_conditions=exposure_conditions,
                                            co2_scenario=co2_scenario,
                                            lambda3_fixed=lambda3_fixed,
                                            lambda4_fixed=lambda4_fixed,
                                            n_glam_samples=n_glam_samples,
                                            input_dir='.',
                                            output_dir='.',
                                         )

lambda_df = result['lambda_df']
samples   = result['samples']
print(f"\nSamples shape: {samples.shape}  (n_glam_samples x len(times))")
lambda_df

GENERATING THE RUL DATASET

----------------------------------------
GENERATING RUL DATASET AT fck=25.0, rh=55.0, cov=25.0
----------------------------------------
  lambda 3 fixed at 0.1983, lambda 4 fixed at 0.1300
  t = 0.0: lambda 1 = 24.810, lambda 2 = 0.393, sample mean = 24.946, sample std = 3.585
  t = 5.3: lambda 1 = 21.663, lambda 2 = 0.352, sample mean = 21.815, sample std = 4.002
  t = 10.5: lambda 1 = 18.535, lambda 2 = 0.308, sample mean = 18.708, sample std = 4.568
  t = 15.8: lambda 1 = 14.823, lambda 2 = 0.291, sample mean = 15.006, sample std = 4.838
  t = 21.1: lambda 1 = 10.812, lambda 2 = 0.277, sample mean = 11.005, sample std = 5.086
  t = 26.3: lambda 1 = 7.332, lambda 2 = 0.261, sample mean = 7.537, sample std = 5.390
  t = 31.6: lambda 1 = 5.300, lambda 2 = 0.244, sample mean = 5.519, sample std = 5.759
  t = 36.8: lambda 1 = 3.439, lambda 2 = 0.233, sample mean = 3.669, sample std = 6.048
  t = 42.1: lambda 1 = 2.266, lambda 2 = 0.221, sample mean = 2.508, sa

,fck,rh,cov,Time (years),lambda 1,lambda 2,lambda 3,lambda 4
0,25.0,55.0,25.0,0.000000,24.809648,0.392594,0.198312,0.130039
1,25.0,55.0,25.0,5.263158,21.662538,0.351681,0.198312,0.130039
2,25.0,55.0,25.0,10.526316,18.534814,0.308140,0.198312,0.130039
3,25.0,55.0,25.0,15.789474,14.822707,0.290902,0.198312,0.130039
4,25.0,55.0,25.0,21.052632,10.811846,0.276716,0.198312,0.130039
5,25.0,55.0,25.0,26.315789,7.331875,0.261123,0.198312,0.130039
6,25.0,55.0,25.0,31.578947,5.300013,0.244408,0.198312,0.130039
7,25.0,55.0,25.0,36.842105,3.439305,0.232728,0.198312,0.130039
8,25.0,55.0,25.0,42.105263,2.265926,0.220727,0.198312,0.130039
9,25.0,55.0,25.0,47.368421,1.086627,0.210085,0.198312,0.130039


## 4. Sanity check

In [4]:
summary = pd.DataFrame({
                           'Time (years)': result['times'],
                           'Sample mean':  samples.mean(axis=0),
                           'Sample std':   samples.std(axis=0),
                           'P(g <= 0)':    (samples <= 0).mean(axis=0),
                        })
summary

,Time (years),Sample mean,Sample std,P(g <= 0)
0,0.000000,24.945812,3.585109,0.00000
1,5.263158,21.814542,4.002181,0.00000
2,10.526316,18.708297,4.567702,0.00000
3,15.789474,15.006470,4.838361,0.00010
4,21.052632,11.005030,5.086415,0.01088
5,26.315789,7.536594,5.390146,0.07832
6,31.578947,5.518733,5.758784,0.17324
7,36.842105,3.669002,6.047781,0.28108
8,42.105263,2.508112,6.376599,0.35650
9,47.368421,1.341081,6.699629,0.43068


In [5]:
summary_5mm = pd.DataFrame({
                              'Time (years)': result['times'],
                              'Sample mean':  samples.mean(axis=0),
                              'Sample std':   samples.std(axis=0),
                              'P(g <= 5)':    (samples <= 5.0).mean(axis=0),
                           })
summary_5mm

,Time (years),Sample mean,Sample std,P(g <= 5)
0,0.000000,24.945812,3.585109,0.00000
1,5.263158,21.814542,4.002181,0.00000
2,10.526316,18.708297,4.567702,0.00020
3,15.789474,15.006470,4.838361,0.01456
4,21.052632,11.005030,5.086415,0.11902
5,26.315789,7.536594,5.390146,0.32866
6,31.578947,5.518733,5.758784,0.47134
7,36.842105,3.669002,6.047781,0.59242
8,42.105263,2.508112,6.376599,0.65716
9,47.368421,1.341081,6.699629,0.71046
